In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from collections import defaultdict
import pickle

# Carregar histórico e clusters
df_historico = pd.read_csv("arquivos/historico.csv", dtype={
    "userId": "string", "page": "string", "timestampHistory": "string",
    "numberOfClicks": "int", "timeOnPage": "int",
    "scrollPercentage": "float", "pageVisitsCount": "int"
})
df_clusters = pd.read_csv("arquivos/noticias_clusterizadas.csv", dtype={"page": "string", "cluster": "int"})

In [7]:
# Processar histórico (merge com clusters e ordena)
df_historico["timestampHistory"] = pd.to_datetime(df_historico["timestampHistory"], errors="coerce")
df_historico = df_historico.merge(df_clusters[["page", "cluster"]], left_on="page", right_on="page", how="left")
df_historico = df_historico.sort_values(["userId", "timestampHistory"])

# Contar páginas por usuário e filtrar apenas usuários com histórico ≥ 3 páginas
df_user_page_count = df_historico.groupby("userId")["page"].count().reset_index()
df_user_page_count.columns = ["userId", "qtde_paginas"]
usuarios_validos = df_user_page_count[df_user_page_count["qtde_paginas"] >= 3]["userId"]
df_historico = df_historico[df_historico["userId"].isin(usuarios_validos)]

# Criar colunas para os clusters das últimas 3 páginas
df_historico["prev_cluster_1"] = df_historico.groupby("userId")["cluster"].shift(1).fillna(-1).astype(int)
df_historico["prev_cluster_2"] = df_historico.groupby("userId")["cluster"].shift(2).fillna(-1).astype(int)
df_historico["prev_cluster_3"] = df_historico.groupby("userId")["cluster"].shift(3).fillna(-1).astype(int)

# Criar next_cluster e remover linhas sem próxima página
df_historico["next_cluster"] = df_historico.groupby("userId")["cluster"].shift(-1)
df_historico = df_historico.dropna(subset=["next_cluster"])
df_historico["next_cluster"] = df_historico["next_cluster"].astype(int)

# Criar df_user_last (última linha de cada user, trazendo as últimas 3 páginas)
df_user_last = df_historico.groupby("userId").tail(1)[[
    "userId", "cluster", "prev_cluster_1", "prev_cluster_2", "prev_cluster_3"
]]
df_user_last = df_user_last.rename(columns={"cluster": "last_cluster"})

# Contagem de visitas por cluster
cluster_ids = df_historico["cluster"].dropna().unique()
df_cluster_count = df_historico.groupby(["userId", "cluster"]).size().unstack(fill_value=0).reset_index()
df_cluster_count.columns = ["userId"] + [f"visitas_cluster_{int(c)}" for c in cluster_ids]

# Tempo médio e scroll médio por cluster
df_cluster_metrics = df_historico.groupby(["userId", "cluster"]).agg(
    tempo_medio=("timeOnPage", "mean"),
    scroll_medio=("scrollPercentage", "mean")
).unstack(fill_value=0).reset_index()

df_cluster_metrics.columns = ["userId"] + [
    f"{col}_{int(cluster_id)}" for col, cluster_id in df_cluster_metrics.columns[1:]
]

In [8]:
# Consolidar features por usuário
df_user_features = (
    df_user_last
    .merge(df_cluster_count, on="userId", how="left").fillna(0)
    .merge(df_cluster_metrics, on="userId", how="left").fillna(0)
)

# Garantir colunas de visitas como inteiras
for col in df_user_features.columns:
    if col.startswith("visitas_cluster_"):
        df_user_features[col] = df_user_features[col].astype(int)

# Juntar com target (next_cluster)
df_user_history = df_user_features.merge(
    df_historico.groupby("userId").tail(1)[["userId", "next_cluster"]],
    on="userId"
)

# Definir features (incluindo os 3 últimos clusters)
features = ["last_cluster", "prev_cluster_1", "prev_cluster_2", "prev_cluster_3"] + [
    col for col in df_user_features.columns if col.startswith(("visitas_cluster_", "tempo_medio_", "scroll_medio_"))
]

X = df_user_history[features]
y = df_user_history["next_cluster"]

In [9]:
# Normalizar features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Separar treino e teste
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, stratify=y, random_state=42)

# Aplicar SMOTE para balancear classes
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)

In [10]:
# Modelos a serem treinados
modelos = {
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric="mlogloss"),
    "LightGBM": LGBMClassifier(n_estimators=100, random_state=42)
}

# Armazenar resultados
resultados = defaultdict(dict)

# Treinar e avaliar
for nome, modelo in modelos.items():
    print(f"\nTreinando modelo: {nome}")
    modelo.fit(X_res, y_res)
    y_pred = modelo.predict(X_test)

    # Relatório de métricas
    relatorio = classification_report(y_test, y_pred, output_dict=True)
    resultados[nome]["accuracy"] = relatorio["accuracy"]
    resultados[nome]["macro_f1"] = relatorio["macro avg"]["f1-score"]

    print(f"\nRelatório de Classificação - {nome}:")
    print(classification_report(y_test, y_pred))

# Comparar resultados entre modelos
df_resultados = pd.DataFrame(resultados).T.reset_index().rename(columns={"index": "modelo"})
print("\nComparação Final entre Modelos:")
print(df_resultados)

# Selecionar melhor modelo (com maior macro F1)
melhor_modelo = df_resultados.sort_values("macro_f1", ascending=False).iloc[0]
print(f"\nMelhor modelo: {melhor_modelo['modelo']} com Macro F1 = {melhor_modelo['macro_f1']:.4f}")


Treinando modelo: RandomForest

Relatório de Classificação - RandomForest:
              precision    recall  f1-score   support

           0       0.15      0.14      0.14      2023
           1       0.12      0.39      0.18        57
           2       0.33      0.32      0.32      8492
           3       0.15      0.17      0.16      1489
           4       0.03      0.13      0.04        47
           5       0.43      0.42      0.43     16653
           6       0.18      0.24      0.21      1523
           7       0.40      0.40      0.40     16754

    accuracy                           0.37     47038
   macro avg       0.22      0.28      0.24     47038
weighted avg       0.37      0.37      0.37     47038


Treinando modelo: XGBoost


c:\Projetos\project_globo\modelo\venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [16:41:02] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



Relatório de Classificação - XGBoost:
              precision    recall  f1-score   support

           0       0.24      0.15      0.19      2023
           1       0.06      0.37      0.10        57
           2       0.39      0.32      0.35      8492
           3       0.24      0.21      0.22      1489
           4       0.01      0.28      0.02        47
           5       0.45      0.41      0.43     16653
           6       0.25      0.25      0.25      1523
           7       0.42      0.47      0.44     16754

    accuracy                           0.40     47038
   macro avg       0.26      0.31      0.25     47038
weighted avg       0.40      0.40      0.40     47038


Treinando modelo: LightGBM
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015033 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7140
[LightGBM] [Info] Numb

c:\Projetos\project_globo\modelo\venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Relatório de Classificação - LightGBM:
              precision    recall  f1-score   support

           0       0.24      0.17      0.20      2023
           1       0.06      0.37      0.10        57
           2       0.40      0.30      0.34      8492
           3       0.24      0.22      0.23      1489
           4       0.01      0.23      0.02        47
           5       0.46      0.41      0.43     16653
           6       0.27      0.28      0.27      1523
           7       0.42      0.49      0.45     16754

    accuracy                           0.40     47038
   macro avg       0.26      0.31      0.26     47038
weighted avg       0.41      0.40      0.40     47038


Comparação Final entre Modelos:
         modelo  accuracy  macro_f1
0  RandomForest  0.368149  0.235959
1       XGBoost  0.395914  0.250703
2      LightGBM  0.399783  0.255025

Melhor modelo: LightGBM com Macro F1 = 0.2550


In [11]:
# Salvar melhor modelo e scaler
with open("pkls/modelo_final.pkl", "wb") as f:
    pickle.dump(modelos[melhor_modelo['modelo']], f)

with open("pkls/scaler_final.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("\nModelo final e scaler salvos com sucesso!")


Modelo final e scaler salvos com sucesso!
